# Lab: Concatenate attention for German-to-English translation (Solved)


In [ ]:
import random
import re
from collections import Counter
from pathlib import Path
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader

SEED = 17
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_MODE = "multi30k"
RUN_FULL_TRAINING = True
MAX_TRAIN_PAIRS = 12000
OUTPUT_DIR = Path("seq2seq_attention_outputs")
print("PyTorch", torch.__version__, "device", device, "mode", DATA_MODE)


### Exercise 1 Answer
Given batch size $B=3$, padded source length $T_x=12$, target length $T_y=9$, hidden width $H=256$, and vocabulary $V_y=4000$:
*   **Encoder outputs:** $(3, 12, 256)$
*   **Top decoder hidden state:** $(1, 3, 256)$
*   **Attention scores:** $(3, 12)$
*   **Attention weights:** $(3, 12)$
*   **Context:** $(3, 256)$
*   **Training logits:** $(3, 8, 4000)$
*   **Why target at step $t$ cannot be input at step $t$:** Passing the target token at step $t$ as the input at step $t$ would expose the exact token the model is trying to predict. During autoregressive generation, the model must predict step $t$ using only the context of steps $< t$.


In [ ]:
class ConcatAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.alignment = nn.Linear(2 * hidden_dim, hidden_dim)
        self.score = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, query, encoder_outputs, src_mask):
        # Repeat query over source positions: (B, H) -> (B, 1, H) -> (B, T_x, H)
        T_x = encoder_outputs.size(1)
        query_repeated = query.unsqueeze(1).expand(-1, T_x, -1)
        
        # Concatenate and score: (B, T_x, 2H) -> (B, T_x, H) -> (B, T_x, 1)
        concat = torch.cat((query_repeated, encoder_outputs), dim=2)
        energy = torch.tanh(self.alignment(concat))
        scores = self.score(energy).squeeze(2)
        
        # Mask padding before softmax
        scores = scores.masked_fill(~src_mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        
        # Form the weighted sum of encoder outputs: (B, 1, T_x) @ (B, T_x, H) -> (B, H)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        
        return context, weights

ATTENTION_READY = True


In [ ]:
def beam_translate(model, source_text, beam_width=3, max_steps=40, length_alpha=0.6):
    model.eval()
    src_ids = [SOS] + [src_vocab.get(t, UNK) for t in tokenize(source_text)] + [EOS]
    src = torch.tensor(src_ids, device=device).unsqueeze(0)
    lengths = torch.tensor([len(src_ids)], device=device)

    with torch.no_grad():
        encoder_outputs, (hidden, cell) = model.encoder(src, lengths)

    # Beams hold tuples of: (token_ids, log_score, hidden, cell, ended_flag)
    beams = [([SOS], 0.0, hidden, cell, False)]

    for _ in range(max_steps):
        new_beams = []
        for seq, score, h, c, ended in beams:
            if ended:
                new_beams.append((seq, score, h, c, ended))
                continue

            prev = torch.tensor([seq[-1]], device=device)
            with torch.no_grad():
                logits, next_h, next_c, _ = model.decoder(
                    prev, h, c, encoder_outputs, src.ne(PAD)
                )
            
            log_probs = torch.log_softmax(logits.squeeze(0), dim=-1)
            top_log_probs, top_indices = log_probs.topk(beam_width)

            for i in range(beam_width):
                next_token = top_indices[i].item()
                next_score = score + top_log_probs[i].item()
                is_eos = (next_token == EOS)
                new_beams.append((seq + [next_token], next_score, next_h, next_c, is_eos))

        # Rank with length penalty: ((5 + generated_length) / 6) ** length_alpha
        def get_score(beam):
            seq, score, _, _, _ = beam
            gen_len = len(seq) - 1 # don't count <sos> in generated length
            if gen_len == 0: gen_len = 1
            return score / (((5 + gen_len) / 6) ** length_alpha)

        beams = sorted(new_beams, key=get_score, reverse=True)[:beam_width]

        if all(beam[4] for beam in beams):
            break

    best_seq = beams[0][0]
    best_seq = [tok for tok in best_seq if tok not in (SOS, EOS)]
    return [tgt_itos.get(idx, "<unk>") for idx in best_seq]

BEAM_READY = True


### Exercise 5 & 6 Answers

**Exercise 5: Perplexity vs. BLEU:** A model can have lower (better) perplexity but lower (worse) BLEU because perplexity evaluates how well the model predicts exact reference tokens given perfect previous reference tokens (teacher-forced). BLEU evaluates the generated translation end-to-end without reference context. A model might be highly confident in assigning probabilities, but fail at autoregressive generation due to cascading errors over long sequences.

**Greedy vs. Beam on Metrics:** Changing the decoding strategy changes BLEU because it alters the sequence of words actually generated by the model. It cannot change perplexity because perplexity is a mathematical property calculated strictly over the evaluation loss against the ground truth labels, completely independent of the inference generation strategy.

**Exercise 6: Attention Map Interpretation:**
*   **Comparing attention vs sequence-level attention:** In this Seq2Seq model, attention is recomputed at *every decoder step*, allowing the network to continuously scan the German source and shift its focal point as it constructs the English sentence. In a classifier (like the Disaster Tweets sequence-to-one model), attention is typically calculated once over the entire sequence to pool a single contextual representation.
*   **Limitation of Interpretation:** A high attention weight does not definitively prove linguistic alignment (i.e., that word X translates to word Y). The model could be looking at an earlier source token to extract syntactic cues (like pluralization or tense markers) needed to properly predict the next English token, rather than translating the word it is focusing on.
